Read the [accompanying blog post](https://acytoo.com/ladder/llm-tut-4-tokenizer-position-embedding/) for explanations and experiment results.

# Model implementation in Pytorch
require Pytorch > 2.10

https://github.com/karpathy/nanoGPT

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

### Model blocks

In [ ]:
def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1).flatten(-2)


class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

        # Precompute Rotary Position Embedding (RoPE) cos/sin tables
        head_dim = config.n_embd // config.n_head
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        t = torch.arange(config.block_size, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)
        self.register_buffer("rope_cos", freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer("rope_sin", freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # Apply RoPE to Query and Key before dot-product attention
        cos = self.rope_cos[:, :, :T, :].to(q.dtype)
        sin = self.rope_sin[:, :, :T, :].to(q.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


### Model

In [ ]:
@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float
    bias: bool

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # with weight tying when using torch.compile() some warnings get generated:
        # "UserWarning: functional_call was passed multiple values for tied weights.
        # This behavior is deprecated and will be an error in future versions"
        # not 100% sure what this is, so far seems to be harmless. TODO investigate
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"

        # forward the GPT model itself (RoPE is applied inside CausalSelfAttention)
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        x = self.transformer.drop(tok_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss

    def crop_block_size(self, block_size):
        # model surgery to decrease the block size if necessary
        assert block_size <= self.config.block_size
        self.config.block_size = block_size

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        use_fused = (device_type == 'cuda')
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# Train our new model

### Prepare data: Chinese law

In [ ]:
from pathlib import Path

if not Path("chinese-law-corpus").exists():
    !git clone --depth 1 -q https://github.com/lttxzmj/chinese-law-corpus.git

CORPUS_REVISION = "c1eb6e74adac5c36e9c28dc8e027d19711ab369b"
!git -C chinese-law-corpus fetch --depth 1 origin {CORPUS_REVISION}
!git -C chinese-law-corpus checkout --detach {CORPUS_REVISION}


remote: Total 0 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
From https://github.com/lttxzmj/chinese-law-corpus
 * branch            c1eb6e74adac5c36e9c28dc8e027d19711ab369b -> FETCH_HEAD
HEAD is now at c1eb6e7 Resync the corpus and refresh its generated counts


In [ ]:
import json
import random
from pathlib import Path
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
corpus_dir = Path("chinese-law-corpus")


def to_clean_str(val) -> str:
    """Safely convert str, list of paragraphs, or dict into a clean string."""
    if val is None:
        return ""
    if isinstance(val, str):
        return val.strip()
    if isinstance(val, list):
        parts = [to_clean_str(item) for item in val]
        return "\n".join(p for p in parts if p)
    if isinstance(val, dict):
        for k in ("text", "content", "paragraph", "value"):
            if k in val:
                return to_clean_str(val[k])
        return "\n".join(to_clean_str(v) for v in val.values() if v)
    return str(val).strip()


# 1. Load Core Foundational Statutes
CORE_LAW_KEYWORDS = (
    "民法典", "刑法", "公司法", "劳动合同法", "民事诉讼法",
    "刑事诉讼法", "行政诉讼法", "治安管理处罚法", "消费者权益保护法",
    "著作权法", "专利法", "商标法",
)

statute_chunks = []
law_count = 0
article_count = 0

for p in sorted((corpus_dir / "laws").glob("*.json")):
    if p.name == "index.json":
        continue
    data = json.loads(p.read_text(encoding="utf-8"))
    if not isinstance(data, dict) or "articles" not in data:
        continue
    title = to_clean_str(data.get("title", ""))
    if not any(k in title for k in CORE_LAW_KEYWORDS):
        continue
    status = to_clean_str(data.get("status") or data.get("effectiveStatus", "")).lower()
    if status not in ("effective", "现行有效", "有效"):
        continue

    articles = data.get("articles", [])
    if not articles:
        continue
    law_count += 1
    # Group every 10 consecutive articles into a chunk for clean train/val splitting
    buf = []
    for art in articles:
        if not isinstance(art, dict):
            continue
        num = to_clean_str(art.get("displayNumber") or art.get("number"))
        txt = to_clean_str(art.get("text") or art.get("content"))
        if not txt:
            continue
        article_count += 1
        buf.append(f"《{title}》{num}：{txt}")
        if len(buf) == 10:
            statute_chunks.append("\n".join(buf) + "\n\n")
            buf = []
    if buf:
        statute_chunks.append("\n".join(buf) + "\n\n")

# 2. Load Supreme People's Court Guiding Cases (指导性案例)
text_chunks = list(statute_chunks)
case_count = 0
for p in sorted((corpus_dir / "guiding-cases").glob("*.json")):
    if p.name == "index.json":
        continue
    c = json.loads(p.read_text(encoding="utf-8"))
    if not isinstance(c, dict):
        continue
    title = to_clean_str(c.get("title") or p.stem)
    parts = [title]
    for field in ("facts", "gist", "result", "reasoning"):
        value = to_clean_str(c.get(field))
        if value:
            parts.append(value)
    if len(parts) > 1:
        text_chunks.append("\n".join(parts) + "\n\n")
        case_count += 1

text = "".join(text_chunks)
assert text, "No text loaded. Check the corpus directory."


In [ ]:
print("len", len(text))
print(text[:80])
print(text[-80:])

len 1604806
《中华人民共和国行政诉讼法》第一条：为保证人民法院公正、及时审理行政案件，解决行政争议，保护公民、法人和其他组织的合法权益，监督行政机关依法行使职权，根据宪法，
共利益。在此情形下，洪振快有能力控制文章所可能产生的损害后果而未控制，仍以既有的状态发表，在主观上显然具有过错。
(生效裁判审判人员：王平、何江恒、赵胤晨)




### Tokenizer (BPE)

In [ ]:
import re
import heapq
from collections import Counter, defaultdict
from functools import lru_cache

PAT = re.compile(r"[\u4e00-\u9fff]+|[0-9]+|[a-zA-Z]+|[^\s\u4e00-\u9fff0-9a-zA-Z]+|\s+")

all_chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(all_chars)}
itos = {i: ch for ch, i in stoi.items()}

target_vocab_size = 4096
chunk_counts = Counter(PAT.findall(text))
words = [[stoi[ch] for ch in chunk] for chunk in chunk_counts.keys()]
counts = list(chunk_counts.values())

pair_counts = defaultdict(int)
pair_to_words = defaultdict(set)
for wid, (w, c) in enumerate(zip(words, counts)):
    for pair in zip(w, w[1:]):
        pair_counts[pair] += c
        pair_to_words[pair].add(wid)

heap = [(-cnt, pair) for pair, cnt in pair_counts.items()]
heapq.heapify(heap)

merges = {}  # (id_a, id_b) -> new_id
next_id = len(all_chars)
while next_id < target_vocab_size and heap:
    neg_cnt, best_pair = heapq.heappop(heap)
    if pair_counts.get(best_pair, 0) != -neg_cnt or -neg_cnt <= 0:
        continue
    merges[best_pair] = next_id
    itos[next_id] = itos[best_pair[0]] + itos[best_pair[1]]
    a, b = best_pair
    affected = list(pair_to_words.pop(best_pair))
    changed = set()
    for wid in affected:
        w = words[wid]
        c = counts[wid]
        for pair in zip(w, w[1:]):
            pair_counts[pair] -= c
            if pair != best_pair:
                pair_to_words[pair].discard(wid)
                changed.add(pair)
        new_w = []
        i = 0
        while i < len(w):
            if i + 1 < len(w) and w[i] == a and w[i + 1] == b:
                new_w.append(next_id)
                i += 2
            else:
                new_w.append(w[i])
                i += 1
        words[wid] = new_w
        for pair in zip(new_w, new_w[1:]):
            pair_counts[pair] += c
            pair_to_words[pair].add(wid)
            changed.add(pair)
    for p in changed:
        if pair_counts[p] > 0:
            heapq.heappush(heap, (-pair_counts[p], p))
    next_id += 1

vocab_size = next_id


@lru_cache(maxsize=None)
def encode_chunk(chunk: str) -> tuple[int, ...]:
    ids = [stoi[ch] for ch in chunk if ch in stoi]
    while len(ids) >= 2:
        best_pair = min(
            ((ids[i], ids[i + 1]) for i in range(len(ids) - 1)),
            key=lambda p: merges.get(p, float("inf")),
        )
        if best_pair not in merges:
            break
        new_id = merges[best_pair]
        a, b = best_pair
        new_ids = []
        i = 0
        while i < len(ids):
            if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                new_ids.append(new_id)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        ids = new_ids
    return tuple(ids)


def encode_text(s: str) -> list[int]:
    out = []
    for chunk in PAT.findall(s):
        out.extend(encode_chunk(chunk))
    return out


def decode_text(tokens: list[int]) -> str:
    return "".join(itos[t] for t in tokens)

# check bpe result
base_n = len(all_chars)
print(f"Base chars: {base_n:,} | BPE merges: {len(merges):,} | Final vocab_size: {vocab_size:,}")
print("First 25 merged tokens:", [itos[base_n + i] for i in range(25)])
demo = "《中华人民共和国民法典》第五百七十八条：合同的当事人约定代表人或者履行义务的真实意思表示"
demo_ids = encode_text(demo)
print(f"Demo ({len(demo)} chars -> {len(demo_ids)} tokens):", " / ".join(itos[i] for i in demo_ids))

Base chars: 3,062 | BPE merges: 1,034 | Final vocab_size: 4,096
First 25 merged tokens: ['人民', '公司', '法院', '或者', '人民法院', '和国', '中华', '共和国', '人民共和国', '中华人民共和国', '规定', '应当', '诉讼', '20', '申请', '可以', '适用', '最高', '行为', '民事', '执行', '关于', '第一', '最高人民法院', '诉讼法']
Demo (44 chars -> 18 tokens): 《 / 中华人民共和国民法典 / 》 / 第五百 / 七 / 十八条 / ： / 合同的 / 当事人 / 约定 / 代表 / 人或者 / 履行 / 义务的 / 真实 / 意 / 思 / 表示


### Prepare train/val data

In [ ]:
train_text = "".join(chunk for i, chunk in enumerate(text_chunks) if i % 10 != 9)
val_text = "".join(chunk for i, chunk in enumerate(text_chunks) if i % 10 == 9)

train_data = torch.tensor(encode_text(train_text), dtype=torch.long)
val_data = torch.tensor(encode_text(val_text), dtype=torch.long)

print(f"Loaded {law_count} legal documents ({article_count:,} articles) + {case_count} guiding cases")
print(f"Vocabulary: {vocab_size:,} characters")
print(f"Train: {len(train_data):,} tokens | Val: {len(val_data):,} tokens")


Loaded 93 legal documents (5,761 articles) + 278 guiding cases
Vocabulary: 4,096 characters
Train: 917,987 tokens | Val: 102,782 tokens


### Model instance

In [ ]:
from contextlib import nullcontext

block_size = 512
batch_size = 32
eval_interval = 200
eval_iters = 5

torch.manual_seed(1337)
train_rng = random.Random(1337)

def get_batch(split: str, rng: random.Random = None):
    rng = train_rng if rng is None else rng
    data = train_data if split == "train" else val_data
    starts = [rng.randrange(len(data) - block_size) for _ in range(batch_size)]
    x = torch.stack([data[i : i + block_size] for i in starts])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)


# Baseline 2019 GPT-2 (nanoGPT) architecture: 6 layers, 6 heads, 384 dim (~11.9M params)
config = GPTConfig(
    block_size=block_size,
    vocab_size=vocab_size,
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.1,
    bias=False,
)
model = GPT(config).to(device)
optimizer = model.configure_optimizers(
    weight_decay=0.1,
    learning_rate=3e-4,
    betas=(0.9, 0.95),
    device_type=device,
)

amp_enabled = device == "cuda"
dtype = torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)


def amp_context():
    if amp_enabled:
        return torch.amp.autocast(device_type="cuda", dtype=dtype)
    return nullcontext()


@torch.no_grad()
def estimate_loss():
    was_training = model.training
    model.eval()
    results = {}
    val_rng = random.Random(2026)
    for split in ("train", "val"):
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split, val_rng)
            with amp_context():
                _, loss = model(xb, yb)
            losses.append(loss.item())
        results[split] = sum(losses) / len(losses)
    model.train(was_training)
    return results

number of parameters: 12.19M
num decayed parameter tensors: 25, with 12,189,696 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True


### Train

In [ ]:
import time

max_iters = 1200
model.train()
t0 = time.time()

for step in range(1, max_iters + 1):
    xb, yb = get_batch("train")
    optimizer.zero_grad(set_to_none=True)
    with amp_context():
        _, loss = model(xb, yb)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if step == 1 or step % eval_interval == 0 or step == max_iters:
        dt = time.time() - t0
        val_losses = estimate_loss()
        val_log = " | ".join(f"{k}={v:.3f}" for k, v in val_losses.items())
        print(f"step {step:4d} ({dt:5.1f}s) | batch_loss={loss.item():.3f} | {val_log}")

step    1 (  1.7s) | batch_loss=8.394 | train=8.023 | val=8.021
step  200 ( 31.4s) | batch_loss=4.596 | train=4.583 | val=4.902
step  400 ( 64.1s) | batch_loss=3.973 | train=3.692 | val=4.304
step  600 (100.9s) | batch_loss=3.264 | train=3.164 | val=4.056
step  800 (140.1s) | batch_loss=2.893 | train=2.732 | val=3.894
step 1000 (177.4s) | batch_loss=2.419 | train=2.385 | val=3.843
step 1200 (216.2s) | batch_loss=2.471 | train=2.086 | val=3.820


### Inference

In [ ]:
SEED = 42
TEMPERATURE = 0.7
TOP_K = 30

torch.manual_seed(SEED)


@torch.inference_mode()
def sample_law_text(prompt: str, max_new_tokens: int = 180):
    model.eval()
    ids = torch.tensor([encode_text(prompt)], dtype=torch.long, device=device)
    with amp_context():
        generated = model.generate(
            ids, max_new_tokens=max_new_tokens,
            temperature=TEMPERATURE, top_k=TOP_K,
        )
    return decode_text(generated[0].tolist())


# 1. Test Statute Continuation (Civil Code & Criminal Law)
statute_prompts = [
    "《中华人民共和国民法典》第五百七十七条：当事人一方不履行合同义务",
    "《中华人民共和国刑法》第二百六十四条：盗窃公私财物，",
    "《中华人民共和国劳动合同法》第三十九条：劳动者有下列情形之一的，用人单位可以解除劳动合同：",
    "犯故意伤害罪，判处",
]

for p in statute_prompts:
    print(f"{p}\n===样例输出===\n")
    print(sample_law_text(p), "\n")


《中华人民共和国民法典》第五百七十七条：当事人一方不履行合同义务
===样例输出===

《中华人民共和国民法典》第五百七十七条：当事人一方不履行合同义务，一方应当履行的一方履行债务并向对方履行请求，对方可以请求债务人履行履行对方；对方请求其履行或者履行债务的，人民法院不予支持。
《中华人民共和国民法典》第五百七十九条：一方不履行债务或者履行债务不符合约定的，对方可以请求对方履行或者第三人履行，但是不影响对方的除外。
《中华人民共和国民法典》第五百七十七条：债务人或者第三人在合理期限内履行债务或者不履行债务不符合约定的，可以向债权人请求对方承担违约责任。
《中华人民共和国民法典》第五百七十四条：因债权债务致使不能实现债务或者履行自己的，对方可以请求其承担违约责任。
《中华人民共和国民法典》第五百七十五条：当事人一方未履行债务、履行期限，对方可以请求其承担违约责任。
《中华人民共和国民法典》第五百七十五条：当事人一方不履行债务，一方请求对方履行或者履行请求的一方支付债务的，应当向对方承担违约 

《中华人民共和国刑法》第二百六十四条：盗窃公私财物，
===样例输出===

《中华人民共和国刑法》第二百六十四条：盗窃公私财物，情节恶劣，造成严重后果的，处二年以下有期徒刑或者拘役。
盗窃、抢夺枪支、弹药、爆炸物、毒品、爆炸物、毒品等危险物质罪，对单位判处罚金，并对其直接负责的主管人员和其他直接责任人员，依照前款的规定处罚。
非法采伐、毁坏珍贵树木或者非法占有毒品的，处二年以下有期徒刑、拘役或者管制，并处或者单处罚金；情节严重的，处三年以上七年以下有期徒刑，并处罚金。
明知他人窃取、刺探、收买、非法买卖、运输毒品的，依照前款的规定处罚。
《中华人民共和国刑法》第二百六十七条：明知是盗窃、抢夺、窃取、抢夺毒品后，不办理吸食、注射毒品的，处二年以下有期徒刑或者拘役。
强奸淫幼女的，依照本法第二百六十二条、第二百六十七条的规定定罪处罚。
《中华人民共和国刑法 

《中华人民共和国劳动合同法》第三十九条：劳动者有下列情形之一的，用人单位可以解除劳动合同：
===样例输出===

《中华人民共和国劳动合同法》第三十九条：劳动者有下列情形之一的，用人单位可以解除劳动合同：
（一）用人单位与用人单位协商一致或者协商解决，劳动者协商解决或者解除劳动合同的；
（二）用人单位在劳动合同的；
（